## Train a multi-head linear probe on simulated embeddings
---
This notebook shows how to predict **more than one category per input** with a linear probe. The existing `LinearProbe` in `stm.classify` maps a frozen foundation embedding to a single class (`argmax` over one set of logits). Here each input still has one frozen embedding, but the model also sees categorical covariates and a class name, and it emits one option for every output category (contour, signal type, duration, …).

No audio and no Perch2 ONNX download are required. Embeddings are simulated; swap them for real `FeatureBlock` rows when you train on Raven units.

The trained architecture is `MultiHeadLinearProbe`: one linear head per output category, softmax / cross-entropy inside each head so options in the same category stay mutually exclusive.


### What "more than one category" means
Three setups get mixed together. Only the last one is what this notebook trains.

| Setup | Output for one clip | Loss | Use when |
|---|---|---|---|
| Multiclass (`LinearProbe` + `argmax`) | **One** class name | BCE on a one-hot row, then pick the max | Folder-named Raven units: the clip is `A` *or* `B` |
| Multilabel (`LinearProbe` + sigmoid threshold) | Any subset of labels | Independent BCE per logit | Tags that can co-occur (`boat` and `rain`) |
| **Multi-head** (this notebook) | **One option per category** | Softmax / CE **inside** each head | Structured attributes: contour is `upsweep` *or* `flat`, *and* duration is `short` *or* `long` |

A single flattened softmax over `upsweep, flat, short, long` is the wrong model: it would force the clip to be only one of those four tokens. Independent sigmoids are also wrong if `short` and `long` cannot both be true. Multi-head softmax matches the annotation: each category is its own mutually exclusive choice.


### Inputs and architecture
The foundation model stays frozen. Only a linear map is trained, so this is still a linear probe.

```text
Perch2 embedding (simulated here, 32-d; real Perch2 is 1536-d)
class_name one-hot  ─┐
site one-hot        ─┼── concat ──► head_contour      (softmax over contour options)
season one-hot      ─┘             head_signal_type  (softmax over signal_type options)
                                   head_duration     (softmax over duration options)
```

Simulated labels are constructed so each input channel matters:

- **contour** depends on the species cluster in the embedding (and therefore also on `class_name`)
- **signal_type** depends on `site`
- **duration** depends on `season`

If you drop the categorical one-hots, the last two heads fall to chance. That ablation is at the end of the notebook.

If `class_name` is something you want to *predict* instead of condition on, add it as another output head rather than packing it into the features.


### Vocabularies
Output categories are the heads. Input vocabs (`class_name`, `site`, `season`) are one-hot columns concatenated onto the embedding. Option lists do not need to be the same length.


In [1]:
from __future__ import annotations

import numpy as np
import torch

from stm.classify import build_multihead_model
from stm.train_utils import (
    encode_multihead_targets,
    pack_probe_features,
    packed_feature_dim,
)

EMBED_DIM = 32  # stand-in for Perch2; use stm.classify.PERCH2_EMBEDDING_DIM on real data
N_TRAIN = 800
N_TEST = 200
SEED = 0

INPUT_VOCABS = {
    "class_name": ["humpback", "blue", "fin", "unidentified"],
    "site": ["MARS", "CINMS", "SOCAL"],
    "season": ["spring", "summer", "fall", "winter"],
}

OUTPUT_CATEGORIES = {
    "contour": ["upsweep", "downsweep", "flat", "arc"],
    "signal_type": ["tonal", "pulsed", "mixed"],
    "duration": ["short", "long"],
}

print("packed width", packed_feature_dim(EMBED_DIM, INPUT_VOCABS))
print("heads", {name: len(opts) for name, opts in OUTPUT_CATEGORIES.items()})

packed width 43
heads {'contour': 4, 'signal_type': 3, 'duration': 2}


### Simulate foundation embeddings and labels
Each class name gets a unit-norm prototype vector. Embeddings are that prototype plus Gaussian noise. Output labels are a deterministic function of class, site, and season so a linear model can recover them. Replace this cell with real Perch2 rows and Raven columns when you leave the tutorial.


In [2]:
def simulate_batch(n: int, rng: np.random.Generator, proto: np.ndarray):
    class_names = INPUT_VOCABS["class_name"]
    sites = INPUT_VOCABS["site"]
    seasons = INPUT_VOCABS["season"]

    class_idx = rng.integers(0, len(class_names), size=n)
    site_idx = rng.integers(0, len(sites), size=n)
    season_idx = rng.integers(0, len(seasons), size=n)
    embeddings = proto[class_idx] + 0.08 * rng.normal(0, 1, size=(n, EMBED_DIM)).astype(np.float32)

    categoricals = {
        "class_name": [class_names[i] for i in class_idx],
        "site": [sites[i] for i in site_idx],
        "season": [seasons[i] for i in season_idx],
    }
    labels = {
        "contour": [OUTPUT_CATEGORIES["contour"][i] for i in class_idx],
        "signal_type": [OUTPUT_CATEGORIES["signal_type"][i] for i in site_idx],
        "duration": [OUTPUT_CATEGORIES["duration"][i % 2] for i in season_idx],
    }
    return embeddings, categoricals, labels


rng = np.random.default_rng(SEED)
class_names = INPUT_VOCABS["class_name"]
proto = rng.normal(0, 1, size=(len(class_names), EMBED_DIM)).astype(np.float32)
proto /= np.linalg.norm(proto, axis=1, keepdims=True)
emb_train, cat_train, y_train = simulate_batch(N_TRAIN, rng, proto)
emb_test, cat_test, y_test = simulate_batch(N_TEST, rng, proto)
print(f"train embeddings {emb_train.shape}  test {emb_test.shape}")
print("example row")
print("  class_name", cat_train["class_name"][0], "site", cat_train["site"][0], "season", cat_train["season"][0])
print("  labels", {name: values[0] for name, values in y_train.items()})

train embeddings (800, 32)  test (200, 32)
example row
  class_name unidentified site SOCAL season summer
  labels {'contour': 'arc', 'signal_type': 'mixed', 'duration': 'long'}


### Pack features and encode targets
`pack_probe_features` concatenates the embedding with one-hot columns in `INPUT_VOCABS` insertion order. `encode_multihead_targets` stores one integer per head. Keep the same vocabs at inference or the one-hot columns will shift.


In [3]:
X_train = pack_probe_features(emb_train, cat_train, INPUT_VOCABS)
X_test = pack_probe_features(emb_test, cat_test, INPUT_VOCABS)
Y_train = encode_multihead_targets(y_train, OUTPUT_CATEGORIES)
Y_test = encode_multihead_targets(y_test, OUTPUT_CATEGORIES)
print("X_train", tuple(X_train.shape), "Y_train", tuple(Y_train.shape))
print("one-hot slices: class_name, site, season after the embedding")
print(X_train[0, EMBED_DIM:].tolist())

X_train (800, 43) Y_train (800, 3)
one-hot slices: class_name, site, season after the embedding
[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0]


### Train the multi-head probe
Adam + mean cross-entropy across heads. This is the same optimizer as `build_model` / `LinearProbe`; the only change is one linear layer per category instead of one shared dense row.


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_multihead_model(
    in_features=X_train.shape[1],
    category_options=OUTPUT_CATEGORIES,
    learning_rate=0.05,
    device=str(device),
)
model.vocabs = {name: list(opts) for name, opts in INPUT_VOCABS.items()}

X_train = X_train.to(device)
Y_train = Y_train.to(device)
n = len(X_train)
epochs = 80
batch_size = 64

for epoch in range(epochs):
    perm = torch.randperm(n, device=device)
    epoch_loss = 0.0
    for lo in range(0, n, batch_size):
        idx = perm[lo : lo + batch_size]
        epoch_loss += model.train_step(X_train[idx], Y_train[idx]) * len(idx)
    epoch_loss /= n
    if epoch % max(1, epochs // 8) == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:4d}  loss {epoch_loss:.5f}")

epoch    0  loss 0.60247
epoch   10  loss 0.01017
epoch   20  loss 0.00416
epoch   30  loss 0.00236
epoch   40  loss 0.00155
epoch   50  loss 0.00111
epoch   60  loss 0.00084
epoch   70  loss 0.00066
epoch   79  loss 0.00054


### Decode one option per category
`predict_labels` runs `argmax` **inside each head**, so one clip becomes a dict of category names rather than a single class index.


In [5]:
pred = model.predict_labels(X_test)
print("held-out accuracy by category")
for name, true_values in y_test.items():
    n_ok = sum(row[name] == t for row, t in zip(pred, true_values))
    print(f"  {name:12s}  {n_ok}/{len(pred)} ({n_ok / len(pred):.1%})")

print("\nfirst five test rows")
for i in range(5):
    true = {name: values[i] for name, values in y_test.items()}
    mark = "ok" if pred[i] == true else "MISS"
    print(f"{mark}  in={{class_name={cat_test['class_name'][i]!r}, site={cat_test['site'][i]!r}, season={cat_test['season'][i]!r}}}")
    print(f"     true={true}")
    print(f"     pred={pred[i]}")

held-out accuracy by category
  contour       200/200 (100.0%)
  signal_type   200/200 (100.0%)
  duration      200/200 (100.0%)

first five test rows
ok  in={class_name='fin', site='CINMS', season='fall'}
     true={'contour': 'flat', 'signal_type': 'pulsed', 'duration': 'short'}
     pred={'contour': 'flat', 'signal_type': 'pulsed', 'duration': 'short'}
ok  in={class_name='humpback', site='MARS', season='spring'}
     true={'contour': 'upsweep', 'signal_type': 'tonal', 'duration': 'short'}
     pred={'contour': 'upsweep', 'signal_type': 'tonal', 'duration': 'short'}
ok  in={class_name='humpback', site='CINMS', season='winter'}
     true={'contour': 'upsweep', 'signal_type': 'pulsed', 'duration': 'long'}
     pred={'contour': 'upsweep', 'signal_type': 'pulsed', 'duration': 'long'}
ok  in={class_name='humpback', site='SOCAL', season='winter'}
     true={'contour': 'upsweep', 'signal_type': 'mixed', 'duration': 'long'}
     pred={'contour': 'upsweep', 'signal_type': 'mixed', 'duration':

### Ablation: embeddings only
Retrain with the embedding and no categorical one-hots. Contour should still work (it lives in the class cluster). Signal type and duration should collapse toward chance (1/3 and 1/2), because those labels were generated from `site` and `season`.


In [6]:
X_emb_train = pack_probe_features(emb_train, cat_train, {})
X_emb_test = pack_probe_features(emb_test, cat_test, {})
emb_model = build_multihead_model(
    in_features=X_emb_train.shape[1],
    category_options=OUTPUT_CATEGORIES,
    learning_rate=0.05,
    device=str(device),
)
X_emb_train = X_emb_train.to(device)
for epoch in range(epochs):
    perm = torch.randperm(n, device=device)
    for lo in range(0, n, batch_size):
        idx = perm[lo : lo + batch_size]
        emb_model.train_step(X_emb_train[idx], Y_train[idx])

emb_pred = emb_model.predict_labels(X_emb_test)
print("embeddings-only accuracy (expect contour high; signal_type ~33%; duration ~50%)")
for name, true_values in y_test.items():
    n_ok = sum(row[name] == t for row, t in zip(emb_pred, true_values))
    print(f"  {name:12s}  {n_ok}/{len(emb_pred)} ({n_ok / len(emb_pred):.1%})")

embeddings-only accuracy (expect contour high; signal_type ~33%; duration ~50%)
  contour       200/200 (100.0%)
  signal_type   86/200 (43.0%)
  duration      107/200 (53.5%)


### Save the checkpoint
The payload stores `state_dict`, `category_options` (output heads), and `vocabs` (input one-hots). Reload with `MultiHeadLinearModel.load` and pack new rows with the same vocabs before calling `predict_labels`.


In [7]:
from pathlib import Path
from stm.classify import MultiHeadLinearModel

ckpt = Path("output_multiclass_probe") / "multihead_linear_probe.pt"
model.save(ckpt, vocabs=INPUT_VOCABS)
loaded = MultiHeadLinearModel.load(ckpt, device=str(device))
print("reloaded categories", list(loaded.category_options))
print("reloaded vocabs", list(loaded.vocabs))
print("reload matches", loaded.predict_labels(X_test)[:3] == pred[:3])

Saving model to output_multiclass_probe/multihead_linear_probe.pt
Loaded output_multiclass_probe/multihead_linear_probe.pt categories=['contour', 'signal_type', 'duration'] D=43
reloaded categories ['contour', 'signal_type', 'duration']
reloaded vocabs ['class_name', 'site', 'season']
reload matches True


### Using real Perch2 embeddings
On labeled Raven units the same packing applies:

1. Embed each clip with `stm.embed.Embedding` or `Perch2Extractor` (1536-d, `PERCH2_EMBEDDING_DIM`).
2. Put known covariates in `INPUT_VOCABS` — site, season, recorder, and/or a class name you already know.
3. Put the categories you want to emit in `OUTPUT_CATEGORIES` (the panacetacea-style fields `contour`, `signal_type`, `peak_freq`, `duration`, `harmonics` are one such schema).
4. Call `pack_probe_features` / `encode_multihead_targets` / `build_multihead_model` as above.

If you only need a single folder-name class per clip, keep using `build_model` and `labels_to_one_hot` as in `train_raven_units.ipynb`. Reach for the multi-head probe when one embedding should produce several structured attributes at once.
